## Pilot verification — run BEFORE each public workshop delivery

This notebook validates the hero-query setup against the **baseline dense** model (which should fail) and the **fine-tuned SPLADE** model (which should pass). The lab notebook's narrative hook depends on baseline dense burying the Exact-grade product around rank 7 — if any hero query already returns the Exact in the top 3 from the baseline, swap that hero before going public.

Run order: every cell, top to bottom. Look for `[FAIL EXPECTED]` and `[PASS]` markers in the output.

In [ ]:
from pathlib import Path
import sys
import json

REPO = Path.cwd().resolve().parent  # kernel cwd is notebooks/ on the VM
DATA = REPO / "data"
sys.path.insert(0, str(REPO))

from qdrant_client import QdrantClient, models

from eval import SpladeEncoder
from scripts.setup_collections import (
    COLLECTION,
    DENSE_VECTOR_NAME,
    SPLADE_VECTOR_NAME,
    DENSE_MODEL as DENSE_MODEL_ID,
    SPLADE_FINETUNED_MODEL_DEFAULT,
)

client = QdrantClient(host="localhost", port=6333)
hero_queries = json.loads((DATA / "hero_queries.json").read_text())
qrels        = json.loads((DATA / "qrels_hero.json").read_text())
splade_encoder = SpladeEncoder(SPLADE_FINETUNED_MODEL_DEFAULT, device="cpu")
assert splade_encoder is not None


def retrieved_ids(points):
    return [p.payload.get("product_id", p.id) for p in points]


def hero_topk_check(label, search_fn, k=3, expect_pass=False):
    """For each hero, check whether the Exact-grade product is in the top-k.

    expect_pass=False (baseline dense): [FAIL EXPECTED] is good, [PASS] is the
        one to fix (the hook depends on baseline failing).
    expect_pass=True  (fine-tuned SPLADE): [PASS] is good, [FAIL EXPECTED]
        means the fine-tuned model didn't recover the Exact and the hero
        needs review.
    """
    print(f"=== {label} — top-{k} ===")
    for q in hero_queries:
        hid = q["id"]
        q_qrels = qrels.get(hid, {})
        exact_ids = {pid for pid, grade in q_qrels.items() if grade == "E"}
        if not exact_ids:
            print(f"[SKIP] {hid} — no Exact-grade product in qrels")
            continue
        ids = retrieved_ids(search_fn(q["text"])[:k])
        hit_rank = None
        for rank, pid in enumerate(ids, start=1):
            if pid in exact_ids:
                hit_rank = rank
                break
        if hit_rank is None:
            tag = "[FAIL EXPECTED]" if not expect_pass else "[FAIL EXPECTED]"
            note = "good — drives the hook" if not expect_pass else "fine-tuned should recover the Exact; review this hero"
            print(f"{tag} {hid} — Exact not in top {k} ({note})")
        else:
            tag = "[PASS]"
            note = "" if expect_pass else "consider swapping this hero — hook depends on baseline failing"
            suffix = f" — {note}" if note else ""
            print(f"{tag} {hid} — Exact at rank {hit_rank}{suffix}")


In [ ]:
# Baseline-dense check: expect [FAIL EXPECTED] on every hero. A [PASS] here
# means the baseline already finds the Exact in the top 3 — the workshop
# hook ("the baseline returns a Substitute at rank 1") will not land.
def search_dense(q):
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=q, model=DENSE_MODEL_ID),
        using=DENSE_VECTOR_NAME, limit=3, with_payload=True,
    ).points


hero_topk_check("baseline_dense", search_dense, k=3, expect_pass=False)


In [ ]:
# Fine-tuned SPLADE check: expect [PASS] on every hero. A [FAIL EXPECTED]
# here means the fine-tuned model didn't recover the Exact and the hero
# needs review before going public.
def search_splade(q):
    idx, vals = splade_encoder.encode([q])[0]
    sv = models.SparseVector(
        indices=list(map(int, idx)), values=list(map(float, vals))
    )
    return client.query_points(
        collection_name=COLLECTION,
        query=sv,
        using=SPLADE_VECTOR_NAME, limit=3, with_payload=True,
    ).points


hero_topk_check("splade_finetuned", search_splade, k=3, expect_pass=True)


### Verdict

- **Baseline dense:** every hero should print `[FAIL EXPECTED]`. If any baseline `[PASS]`s, swap that hero before going public — the hook depends on baseline failing.
- **Fine-tuned SPLADE:** every hero should print `[PASS]`. If any prints `[FAIL EXPECTED]`, the fine-tuned model isn't recovering the Exact for that hero — either swap the hero or retrain.
- **`[SKIP]` lines** mean the hero has no Exact-grade product in `qrels_hero.json`; rebuild the eval data before pilot.
